# Expected-GPA Metrics and Nominal Inference

This notebook exposes the adopted expected-individual-GPA target, allocation-distance metrics, equal-person inference, and the 24 conditional power references. These are separate from the historical calibration notebook.

The target minimizes `abs(E[GPA_A] - E[GPA_B])` on the 1001-allocation grid, not the expected absolute realized gap. Ties favor higher combined expected GPA, then closeness to 50/50, then the lower grid index. Maximizing combined GPA is a different operation.

`R_EGPA_pp = (abs(j - 500) - abs(j - j_EGPA)) / 10`: positive values mean the allocation is closer to expected-GPA equality than to equal split. It is not a hidden true-outcome equality measure.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from a repository checkout.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])


## Scope

The full target panel has 27 working initial states and three allocation rules per state; it is deterministic target geometry, not an evaluation of a new acquisition policy. The power route reproduces the 24 declared-assumption cases, including the N versus N-1 checks and retention references.

Person-level inference uses ten main trials per person, nine separate nominal tests at alpha=.05, fixed directional tails, and pointwise 95% intervals. It makes no familywise guarantee. Missing arms, zero standard errors and unsupported projections remain unavailable; supporting analyses cannot rescue a primary test.

The raw-trial input route is synthetic only. It recomputes beliefs and targets from the supplied histories, rather than accepting cached target columns. It is not a validated human-session importer. The full native 400,000-attempt calibration, its independently derived truth tables, and latest-model cap20 compatibility are not reproduced by this notebook.


In [ ]:
import json
from IPython.display import display

RUN = False
MODE = 'smoke'  # 'full' produces all 27 initial target states.
OUTPUT = ROOT / 'results' / 'expected_gpa_reproduction'
RAW_TRIALS = None  # Optional Path to a synthetic raw-trial JSON array.
if MODE not in {'smoke', 'full'}:
    raise ValueError('MODE must be smoke or full')
runner = ROOT / 'scripts' / 'reproduce_expected_gpa.py'
commands = [
    [sys.executable, str(runner), 'targets', '--output', str(OUTPUT / 'targets')],
    [sys.executable, str(runner), 'power', '--output', str(OUTPUT / 'power')],
]
if MODE == 'smoke':
    commands[0] += ['--template', 'C3_s0_g0']
if RAW_TRIALS is not None:
    commands.append([sys.executable, str(runner), 'analyze', '--trials', str(Path(RAW_TRIALS)),
                     '--output', str(OUTPUT / 'analysis')])
for command in commands:
    print(' '.join(command))
    if RUN:
        subprocess.check_call(command, cwd=ROOT)


## Synthetic Trial Input

`analyze` takes a JSON array. Each record has `templateId`, `participantId`, `mainIndex` (1..10), `paid` (ordered `{student: 0|1, report: 1..100}` objects), `shareIndex` (0..1000, or null if no allocation), and `humanObservations: false`. Optional fields are `sessionId`, `sideReversed`, `recordPresent`, and `endpointPresent`.

Template identities come from `scripts.reproduce_expected_gpa.templates()`. The working C1 total centre gap is one minute. Remaining budget and posteriors are reconstructed; cached expected-GPA targets are not inputs. Complete people have exactly ten distinct main indices in one condition/session. Partial records remain explicitly ineligible under the corresponding estimand.


In [ ]:
if RUN:
    targets = json.loads((OUTPUT / 'targets' / 'result.json').read_text())
    power = json.loads((OUTPUT / 'power' / 'result.json').read_text())
    display({'target_states': targets['templates'], 'target_rows': len(targets['records']),
             'power_cases': len(power['rows']), 'final_enrollment_selected': power['finalNSelected']})
    display(targets['records'][:3])
    display([r for r in power['rows'] if r['d'] == .3 and r['targetPower'] == .8
             and r['tail'] == 'one_sided_primary'])
    if RAW_TRIALS is not None:
        analysis = json.loads((OUTPUT / 'analysis' / 'result.json').read_text())
        display(analysis['tests'])
else:
    print('No calculations or result reads; set RUN=True after checking settings.')
